# 실습 4. OpenAI API 챗봇 서비스 — 멀티턴 메모리 (Day 2 / 150분)

> 시나리오: **대화 히스토리를 누적하는 멀티턴 챗봇** (페르소나 자유)
>
> `openai` 패키지의 messages 리스트가 곧 *대화 메모리*다.
> (LangChain 의 ConversationBufferMemory 와 같은 개념을 직접 구현한다.)

## Task
1. 히스토리 챗봇 — 대화 누적 구조 (`/reset` 초기화)
2. 페르소나 변경 (system prompt) · temperature 0 vs 0.7
3. 환각 발견 — 모르는 사실 묻기 → system 으로 줄이기
4. 토큰/비용 모니터링 — 100턴 누적 비용 예측 + trim 전략

## 0) 준비

In [1]:
from dotenv import load_dotenv
from openai import OpenAI

load_dotenv()
client = OpenAI()
MODEL = "gpt-4o-mini"

SYSTEM = "당신은 친절한 AI 어시스턴트입니다. 모르면 '확인 필요'라고만 답하세요."

## 1) Task 1 — 히스토리 챗봇 (대화 누적)

`history` 리스트가 메모리다. 매 호출에 system + 전체 history 를 함께 보낸다.
`chat()` 을 여러 번 호출하면 앞선 대화를 기억한다. `reset()` 으로 초기화.

In [2]:
history = []          # [{"role": "user"/"assistant", "content": ...}]
usage_log = []        # 턴별 토큰 기록 (Task 4)

def reset():
    history.clear()
    usage_log.clear()
    print("(대화 초기화)")

def chat(message: str, temperature: float = 0.3) -> str:
    history.append({"role": "user", "content": message})
    resp = client.chat.completions.create(
        model=MODEL,
        temperature=temperature,
        messages=[{"role": "system", "content": SYSTEM}, *history],
    )
    answer = resp.choices[0].message.content
    history.append({"role": "assistant", "content": answer})
    usage_log.append(resp.usage.total_tokens)
    return answer

In [7]:
reset()
print(chat("조선시대 왕궁은 경복궁이야. 기억해줘."))
print(chat("조선시대 왕궁 이름이 뭐였지?"))   # 앞 턴을 기억하는지 확인

(대화 초기화)
확인 필요.
조선시대의 주요 왕궁은 경복궁, 창덕궁, 창경궁, 덕수궁, 경희궁 등이 있습니다. 가장 대표적인 왕궁은 경복궁입니다.


## 2) Task 2 — 페르소나 변경 · temperature 비교

`SYSTEM` 만 바꿔 말투가 바뀐다. temperature 0(일관) vs 0.7(다양).

In [8]:
SYSTEM = "당신은 매우 엄격하고 격식 있는 사서입니다. 짧고 단정하게 답합니다."
reset()
print("[엄격한 사서]", chat("주말에 뭐하면 좋을까?", temperature=0))

SYSTEM = "당신은 친근한 동네 친구입니다. 반말로 편하게 답합니다."
reset()
print("[친근한 친구]", chat("주말에 뭐하면 좋을까?", temperature=0.7))
# TODO: 같은 질문을 temperature 0 / 0.7 로 3회씩 — 답변 다양성 차이를 메모

(대화 초기화)
[엄격한 사서] 독서, 산책, 영화 관람, 취미 활동 등을 추천합니다.
(대화 초기화)
[친근한 친구] 주말에 친구들이랑 영화 보거나 맛있는 거 먹는 것도 좋고, 날씨 좋으면 공원에서 산책하거나 피크닉 가는 것도 재밌어! 너는 어떤 걸 좋아해?


## 3) Task 3 — 환각 발견 → system 으로 줄이기

존재하지 않는 사실을 물어 *지어내는지* 본다. 그 뒤 system 을 강화해 5회 반복.

In [21]:
SYSTEM = "당신은 친절한 AI 어시스턴트입니다."
reset()
print("[가드 없음]", chat("2020년 발롱도르 수상자가 누구야?"))

SYSTEM = (
    "당신은 정직한 AI 어시스턴트입니다. "
    "확실하지 않거나 모르는 정보는 절대 지어내지 말고 '확인 필요'라고만 답하세요."
)
reset()
print("[가드 강화]", chat("2020년 발롱도르 수상자가 누구야?"))

SYSTEM = (
    "당신은 정직한 AI 어시스턴트입니다. "
    "확실하지 않거나 모르는 정보는 절대 지어내지 말고 '확인 필요'라고만 답하세요."
    "이전의 기록들을 살펴보고 확률적으로 가능성이 높은 답변을 제시하되 그 답변이 추측임을 밝히고 정확도를 알려주세요"
)
reset()
print("[가드 강화 및 추측]", chat("2020년 발롱도르 수상자가 누구야?"))
# TODO: 환각 발견 로그 + system 수정 히스토리를 표로 남긴다 (5회 반복)

(대화 초기화)
[가드 없음] 2020년 발롱도르 수상자는 코로나19 팬데믹으로 인해 시상식이 취소되었기 때문에 수상자가 발표되지 않았습니다. 발롱도르는 1956년 시작된 이후 처음으로 2020년에는 수여되지 않았습니다.
(대화 초기화)
[가드 강화] 2020년 발롱도르 수상자는 없습니다. 그 해에는 COVID-19 팬데믹으로 인해 시상식이 취소되었습니다.
(대화 초기화)
[가드 강화 및 추측] 2020년 발롱도르 수상자는 없습니다. 코로나19 팬데믹으로 인해 그 해의 시상식이 취소되었습니다.


In [15]:
print("/reset을 입력하면 대화를 초기화합니다. /exit 또는 빈 줄 입력 시 종료합니다.")
while True:
    user_input = input("User> ").strip()
    if not user_input:
        print("대화를 종료합니다.")
        break
    if user_input.lower() == "/reset":
        reset()
        continue
    if user_input.lower() in {"/exit", "/quit", "bye", "종료"}:
        print("대화를 종료합니다.")
        break
    response = chat(user_input)
    print("Model>", response)


/reset을 입력하면 대화를 초기화합니다. /exit 또는 빈 줄 입력 시 종료합니다.
Model> 테슬라의 CEO는 일론 머스크(Elon Musk)입니다. 하지만 회사의 구조나 경영진이 변동될 수 있으므로, 최신 정보를 확인하는 것이 좋습니다.
Model> 테슬라(Tesla, Inc.)는 전기 자동차, 에너지 저장 장치, 태양광 패널 및 관련 기술을 설계하고 제조하는 회사입니다. 주로 전기 자동차 분야에서 잘 알려져 있으며, 지속 가능한 에너지 솔루션을 제공하는 것을 목표로 하고 있습니다. 또한, 자율주행 기술 개발에도 힘쓰고 있습니다.
(대화 초기화)
대화를 종료합니다.


## 4) Task 4 — 토큰/비용 모니터링 + trim 전략

In [27]:
PRICE = 0.30 / 1_000_000   # 입출력 혼합 평균 가정 단가 ($)

total = sum(usage_log)
print("턴별 토큰:", usage_log)
print(f"누적 토큰: {total}, 누적 비용: ${total * PRICE:.5f}")

# 히스토리는 누적될수록 매 호출 토큰이 선형 증가 → 100턴이면 비용 급증
if usage_log:
    per_turn = total / len(usage_log)
    print(f"600턴 단순 추정: ${per_turn * 100 * PRICE:.4f} (실제로는 더 큼 — 누적 때문)")

def trim(history, keep=6):
    """최근 keep 개 메시지만 유지 — 메모리 trim 전략."""
    return history[-keep:]

def estimate_chat_tokens(messages):
    """현재 messages 기준 토큰 수를 대략 계산한다. 가능하면 tiktoken을 사용한다."""
    try:
        import tiktoken
        enc = tiktoken.encoding_for_model(MODEL)
        tokens = 3
        for msg in messages:
            tokens += 3
            for key, value in msg.items():
                if key == "name":
                    tokens += 1
                elif isinstance(value, str):
                    tokens += len(enc.encode(value))
        return tokens
    except Exception:
        return sum(max(1, len(msg.get("content", "")) // 4) for msg in messages) + 10

keep = 6
before_messages = [{"role": "system", "content": SYSTEM}, *history]
after_history = trim(history, keep=keep)
after_messages = [{"role": "system", "content": SYSTEM}, *after_history]

before_tokens = estimate_chat_tokens(before_messages)
after_tokens = estimate_chat_tokens(after_messages)

print(f"trim 적용 전: 메시지 {len(history)}개, 예상 토큰 {before_tokens}")
print(f"trim 적용 후: 메시지 {len(after_history)}개, 예상 토큰 {after_tokens}")
if before_tokens:
    saved = before_tokens - after_tokens
    print(f"절감 토큰: {saved} ({saved / before_tokens * 100:.1f}%)")

dropped_history = history[:-keep] if len(history) > keep else []
if dropped_history:
    summary_memory = " / ".join(
        f"{msg['role']}: {msg['content'][:40].replace(chr(10), ' ')}"
        + ("..." if len(msg['content']) > 40 else "")
        for msg in dropped_history
    )
else:
    summary_memory = "요약할 만큼 오래된 대화가 아직 없습니다."

print("summary memory:", summary_memory)
print(
    "trade-off: trim은 토큰/지연시간/비용을 줄이지만, 오래된 대화의 세부 맥락을 잃는다. "
    "summary memory는 그 손실을 줄여주지만, 요약을 만드는 추가 비용과 요약 누락/왜곡 가능성이 있다."
)


턴별 토큰: [134]
누적 토큰: 134, 누적 비용: $0.00004
600턴 단순 추정: $0.0040 (실제로는 더 큼 — 누적 때문)
trim 적용 전: 메시지 2개, 예상 토큰 84
trim 적용 후: 메시지 2개, 예상 토큰 84
절감 토큰: 0 (0.0%)
summary memory: 요약할 만큼 오래된 대화가 아직 없습니다.
trade-off: trim은 토큰/지연시간/비용을 줄이지만, 오래된 대화의 세부 맥락을 잃는다. summary memory는 그 손실을 줄여주지만, 요약을 만드는 추가 비용과 요약 누락/왜곡 가능성이 있다.


## 회고 / 산출물
- [ ] 환각 발견 로그 + 수정 히스토리
- [ ] 페르소나별 답변 비교 메모
- [ ] 100턴 누적 비용 예측 + trim/summary 전략 한 줄